# Agent Running Test 
### Prerequisite for LLM used in the agent example
1. Install Ollama from https://ollama.com
2. Start Ollama `ollama run qwen3:4b`

### Prerequisites for a Remote VM
1. Start an SSH server (e.g., `openssh-server`) so you can connect to the VM if needed.
2. Install Python virtual environment support (e.g., `python3-venv`).

In [1]:
example_path = 'agent.py'

In [2]:
import requests
from PIL import Image
import matplotlib.pyplot as plt

def im_show(image_url: str):
    if image_url.startswith("http"):
        image = Image.open(requests.get(image_url, stream=True).raw).convert("RGB")
    else:
        image = Image.open(image_url).convert("RGB")

    plt.imshow(image)
    plt.axis("off")
    plt.show()

## Python Interpreter

In [ ]:
output = !python {example_path}
im_show(output[-1])

## Run the agent and tool teogether locally

In [ ]:
# Removed pip_install parameter from decorators in {example_path}
import ast 
import os
import tempfile


with open(example_path, "r") as f:
    source_code = f.read()
    mod = ast.parse(source_code)

# Remove env keyword argument
for node in ast.walk(mod):
    if isinstance(node, (ast.ClassDef, ast.FunctionDef)):
        for dec in node.decorator_list:
            if isinstance(dec, ast.Call):
                dec.keywords = [kw for kw in dec.keywords if kw.arg != "pip_install"]

with tempfile.NamedTemporaryFile("w", delete=False, suffix=".py") as temp_file:
    temp_file.write(ast.unparse(mod))
    temp_file_path = temp_file.name


output = !single-spec run {temp_file_path}
for line in output:
    print(line)
im_show(output[-1])
os.unlink(temp_file_path)

## Run the agent locally while executing the tool inside a local Python virtual environment.

In [ ]:
output = !single-spec run {example_path}
for line in output:
    print(line)
im_show(output[-1])

## Run the agent locally while executing the tool inside a remote Python virtual environment.

In [ ]:
output = !single-spec run {example_path} -c "exec_nodes_remote.toml"
for line in output:
    print(line)
im_show(output[-2])

## Clear Cache on nodes

In [ ]:
!single-spec clear